# SpeechMatics baseline + submission example

This notebook trains a simple baseline for the SpeechMatics shared task and generates a valid submission file.

The competition data are **not included** in this repository. Download the official files from Codabench and update the paths below.


## Installation

Install the required libraries if they are not already available in your environment.


In [ ]:
# Uncomment if needed
# %pip install pandas numpy scikit-learn librosa tqdm


## Imports and configuration

In [ ]:
from pathlib import Path

import librosa
import numpy as np
pandas_imported = False
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC, SVC
from tqdm import tqdm


In [ ]:
TRAIN_CSV = Path('train/corpus_ironia_iberlef2026_train.csv')
TEST_CSV = Path('test/corpus_ironia_iberlef2026_test.csv')
TRAIN_AUDIO_DIR = Path('train/audios/audios_flac')
TEST_AUDIO_DIR = Path('test/audios/audios_flac')
OUTPUT_CSV = Path('submission_file.csv')
MAX_FEATURES = 10_000


## Load data

The training file must include `id`, `transcripcion` and `label`. The test file must include `id` and `transcripcion`.


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(train_df.head())
print(test_df.head())


## Task 1: text-only baseline

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer='word',
    max_features=MAX_FEATURES,
    lowercase=False,
)

text_x_train_sparse = vectorizer.fit_transform(train_df['transcripcion'].fillna(''))
text_x_test_sparse = vectorizer.transform(test_df['transcripcion'].fillna(''))

text_scaler = MinMaxScaler()
text_x_train = text_scaler.fit_transform(text_x_train_sparse.toarray())
text_x_test = text_scaler.transform(text_x_test_sparse.toarray())

text_classifier = LinearSVC(dual='auto')
text_classifier.fit(text_x_train, train_df['label'])
task1_predictions = text_classifier.predict(text_x_test)

print(task1_predictions[:10])


## Task 2: multimodal baseline

In [ ]:
def extract_mfcc_features(audio_path: Path) -> np.ndarray:
    data, sample_rate = librosa.load(audio_path, sr=None)
    mfcc = librosa.feature.mfcc(y=data, sr=sample_rate)
    return np.mean(mfcc.T, axis=0)


def get_audio_features(df: pd.DataFrame, audio_dir: Path, split_name: str) -> np.ndarray:
    features = []
    for item_id in tqdm(df['id'], desc=f'Extracting {split_name} MFCC features'):
        audio_path = audio_dir / f'{item_id}.flac'
        if not audio_path.exists():
            raise FileNotFoundError(f'Audio file not found: {audio_path}')
        features.append(extract_mfcc_features(audio_path))
    return np.vstack(features)


In [ ]:
labels = sorted(train_df['label'].unique().tolist(), reverse=True)
id_to_label = {idx: label for idx, label in enumerate(labels)}
label_to_id = {label: idx for idx, label in id_to_label.items()}
y_train = train_df['label'].map(label_to_id).to_numpy()

mfcc_x_train = get_audio_features(train_df, TRAIN_AUDIO_DIR, 'training')
mfcc_x_test = get_audio_features(test_df, TEST_AUDIO_DIR, 'test')

audio_scaler = MinMaxScaler()
mfcc_x_train = audio_scaler.fit_transform(mfcc_x_train)
mfcc_x_test = audio_scaler.transform(mfcc_x_test)

x_train = np.concatenate((text_x_train, mfcc_x_train), axis=1)
x_test = np.concatenate((text_x_test, mfcc_x_test), axis=1)

print(x_train.shape, x_test.shape)


In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'gamma': [0.1, 0.01, 0.001],
    'kernel': ['rbf'],
}

grid = GridSearchCV(
    SVC(class_weight='balanced'),
    param_grid,
    refit=True,
    verbose=1,
    n_jobs=-1,
)
grid.fit(x_train, y_train)

print(grid.best_params_)

grid_predictions = grid.predict(x_test)
task2_predictions = [id_to_label[int(label_id)] for label_id in grid_predictions]
print(task2_predictions[:10])


## Create the submission file

In [ ]:
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'label_t1': task1_predictions,
    'label_t2': task2_predictions,
})

submission_df.to_csv(OUTPUT_CSV, index=False)
submission_df.head()
